# Preprocesado de imagen v2
Features concretas de la foto 1: calidad (nitidez, brillo, tamano) y puntajes CLIP contra pares de frases (tierno, jaula, borrosa, persona, varios animales).

In [ ]:
# Kaggle monta los inputs en otra carpeta cuando el notebook se crea por API: armar ../input/<nombre>
import os, glob
def _buscar(nombre, archivo):
    for prof in range(0, 5):
        for d in glob.glob("/kaggle/input/" + "*/" * prof):
            d = d.rstrip("/")
            if os.path.basename(d).endswith(nombre) and os.path.exists(os.path.join(d, archivo)):
                return d
    raise FileNotFoundError(nombre)
_inputs = {"petfinder-adoption-prediction": "train/train.csv"}
if not all(os.path.exists(f"../input/{n}/{a}") for n, a in _inputs.items()):
    os.makedirs("/tmp/lab/input", exist_ok=True)
    os.makedirs("/tmp/lab/work", exist_ok=True)
    for n, a in _inputs.items():
        if not os.path.exists(f"/tmp/lab/input/{n}"):
            os.symlink(_buscar(n, a), f"/tmp/lab/input/{n}")
    os.chdir("/tmp/lab/work")
D = "../input/petfinder-adoption-prediction"
OUT = "/kaggle/working"

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from tqdm import tqdm
fotos = {}
for parte in ("train", "test"):
    fs = sorted(f for f in os.listdir(f"{D}/{parte}_images") if f.endswith("-1.jpg"))
    fotos[parte] = pd.Series([f"{D}/{parte}_images/{f}" for f in fs], index=[f[:-len("-1.jpg")] for f in fs])
    ids = pd.read_csv(f"{D}/{parte}/{parte}.csv")["PetID"]
    print(parte, len(fotos[parte]), "fotos;", fotos[parte].index.isin(ids).sum(), "cruzan con", len(ids), "mascotas")

In [ ]:
# 1) calidad de la foto (sin modelo)
def calidad(path):
    im = cv2.imread(path)
    if im is None:
        return {}
    g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    h, w = g.shape
    return {"img_nitidez": cv2.Laplacian(g, cv2.CV_64F).var(), "img_brillo": g.mean(), "img_contraste": g.std(),
            "img_ancho": w, "img_alto": h, "img_aspecto": w / h}

for parte in ("train", "test"):
    q = pd.DataFrame([calidad(p) for p in tqdm(fotos[parte])], index=fotos[parte].index)
    q.to_parquet(f"{OUT}/img_stats_{parte}.parquet")
    print(parte, q.shape, q.isna().sum().sum())
q.describe().T

In [ ]:
import torch
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:
    (torch.ones(4, 4, device=dev) @ torch.ones(4, 4, device=dev)).sum().item()
except Exception as e:  # GPU que la version de torch ya no soporta
    print("cuda no anda, sigo en cpu:", e)
    dev = "cpu"
print("torch", torch.__version__, dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")

In [ ]:
# 2) CLIP: para cada par (frase, contra-frase), probabilidad de la frase
from transformers import CLIPModel, CLIPProcessor
M = "openai/clip-vit-large-patch14"
clip = CLIPModel.from_pretrained(M).to(dev).eval()
proc = CLIPProcessor.from_pretrained(M)
PARES = {
    "img_tierno": ("a photo of a very cute, adorable pet", "a photo of a pet"),
    "img_jaula": ("a photo of an animal in a cage", "a photo of an animal that is not in a cage"),
    "img_borrosa": ("a blurry, dark, low quality photo", "a sharp, bright, high quality photo"),
    "img_persona": ("a photo of a person holding a pet", "a photo of a pet with no people"),
    "img_varios": ("a photo of several animals together", "a photo of a single animal"),
}
frases = [f for par in PARES.values() for f in par]

def leer(p):
    try:
        return Image.open(p).convert("RGB")
    except Exception:
        return None

B = 64
for parte in ("train", "test"):
    filas = []
    paths = fotos[parte]
    for k in tqdm(range(0, len(paths), B)):
        ims = [(i, leer(p)) for i, p in paths.iloc[k:k + B].items()]
        ims = [(i, im) for i, im in ims if im is not None]
        with torch.no_grad():
            # logits_per_image: similitud coseno * escala de CLIP, (imagenes x frases)
            s = clip(**proc(text=frases, images=[im for _, im in ims], return_tensors="pt", padding=True).to(dev)).logits_per_image
            p = torch.softmax(s.float().view(len(ims), len(PARES), 2), dim=-1)[:, :, 0].cpu().numpy()
        filas += [dict(zip(PARES, p[j]), PetID=i) for j, (i, _) in enumerate(ims)]
    c = pd.DataFrame(filas).set_index("PetID")
    c.to_parquet(f"{OUT}/img_clip_{parte}.parquet")
    print(parte, c.shape)
    for col in PARES:
        print("  ", col, "top 3:", c[col].nlargest(3).round(3).to_dict())
c.describe().T